In [38]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from hmmlearn import hmm
import joblib

# Function to check if an instruction is a known opcode
def is_opcode(instruction):
    known_opcodes = {
        'add', 'sub', 'mul', 'div', 'mov', 'jmp', 'call', 'ret',
        'cmp', 'test', 'inc', 'dec', 'and', 'or', 'xor', 'not',
        'jne', 'je', 'jz', 'jnz', 'jg', 'jl', 'jge', 'jle'
    }
    return instruction in known_opcodes

# Function to read opcode files and extract opcode sequences
def read_opcode_files(file_names, encoding='ISO-8859-1', min_length=5):
    sequences = []
    for file_name in file_names:
        with open(file_name, 'r', encoding=encoding) as file:
            temp_sequence = []
            for line in file:
                line = line.strip()
                if not line or line.startswith(';') or 'segment' in line or 'assume' in line:
                    continue
                opcodes = line.split()
                filtered_opcodes = [opcode.strip() for opcode in opcodes if is_opcode(opcode.strip())]
                if filtered_opcodes:
                    temp_sequence.extend(filtered_opcodes)

            # Break the temp_sequence into chunks of min_length
            for i in range(0, len(temp_sequence), min_length):
                if i + min_length <= len(temp_sequence):
                    sequences.append(temp_sequence[i:i + min_length])
    return sequences

# Function to convert opcodes to numerical features
def opcodes_to_features(sequences):
    unique_opcodes = set(op for seq in sequences for op in seq)
    opcode_to_idx = {opcode: idx for idx, opcode in enumerate(unique_opcodes)}
    
    feature_matrix = []
    for seq in sequences:
        feature_matrix.append([opcode_to_idx[opcode] for opcode in seq if opcode in opcode_to_idx])
    
    return feature_matrix, opcode_to_idx

# Function to smooth the transition probabilities
def smooth_transition_matrix(transmat, alpha=1e-5):
    smoothed_transmat = transmat + alpha
    return smoothed_transmat / smoothed_transmat.sum(axis=1, keepdims=True)

# Function to smooth the emission probabilities
def smooth_emission_probabilities(emissionprob, alpha=1e-5):
    smoothed_emissionprob = emissionprob + alpha
    return smoothed_emissionprob / smoothed_emissionprob.sum(axis=1, keepdims=True)

# Function to train HMM model
def train_hmm_model(sequences, n_components, random_state=42):
    lengths = [len(seq) for seq in sequences]
    if not lengths:
        raise ValueError("No sequences provided for training the HMM model.")

    X = np.concatenate(sequences).reshape(-1, 1)

    # Set a random seed for consistency
    model = hmm.MultinomialHMM(n_components=n_components, n_iter=100, random_state=random_state)
    model.fit(X, lengths)

    # Smooth the transition and emission probabilities after fitting
    model.transmat_ = smooth_transition_matrix(model.transmat_)
    model.emissionprob_ = smooth_emission_probabilities(model.emissionprob_)
    
    return model

# Function to classify unseen opcode sequences
def classify_sequence(model, sequence, epsilon=1e-9):
    sequence_idx = np.array(sequence).reshape(-1, 1)
    log_prob = model.score(sequence_idx)

    # Stabilize log probability with epsilon
    return max(log_prob, np.log(epsilon))

# Function to check if sequences are present in training data
def check_sequences_in_training(training_sequences, sequences_to_check, opcode_to_idx):
    # Create a reverse mapping to go from index back to opcode
    idx_to_opcode = {idx: opcode for opcode, idx in opcode_to_idx.items()}
    
    # Convert training sequences from indices back to opcodes
    training_opcode_sequences = []
    for seq in training_sequences:
        training_opcode_sequences.append([idx_to_opcode[idx] for idx in seq])

    # Check for presence of each sequence
    for seq in sequences_to_check:
        if seq in training_opcode_sequences:
            print(f"Sequence {seq} is present in the training data.")
        else:
            print(f"Sequence {seq} is NOT present in the training data.")


# Function to classify new sequences based on log probabilities
def classify_new_sequence(hmm_model, opcode_to_idx, sequence, threshold):
    features = []
    for opcode in sequence:
        if opcode in opcode_to_idx:
            features.append(opcode_to_idx[opcode])
        else:
            print(f"Warning: Opcode '{opcode}' is not recognized and will be skipped.")
    
    if features:
        log_prob = classify_sequence(hmm_model, features)

        if log_prob <= threshold:
            print(f"Sequence: {sequence} | Log Probability: {log_prob} | Classified as: Malware")
        else:
            print(f"Sequence: {sequence} | Log Probability: {log_prob} | Classified as: Legitimate")
    else:
        print("Sequence contains invalid entries.")

# Main function to load or train the HMM model
def main():
    model_file = 'hmm_model.pkl'
    opcode_to_idx = {}  # Initialize opcode_to_idx as an empty dictionary
    threshold = 0.0
    hmm_model = None
    sequences = []  # Initialize sequences to an empty list

    # Check if the model exists, if yes, load it; if not, train a new one
    if os.path.exists(model_file):
        hmm_model = joblib.load(model_file)
        print("Loaded HMM model from disk.")
        # The opcode_to_idx should be available from previous training
        opcode_to_idx = joblib.load('opcode_to_idx.pkl')
        print("Loaded opcode_to_idx from disk.")
    else:
        file_names = ['IDAN1.asm', 'IDAN2.asm', 'IDAN3.asm']
        sequences = read_opcode_files(file_names)
        print(sequences)
        if not sequences:
            print("No opcode sequences found in provided files.")
            return

        feature_matrix, opcode_to_idx = opcodes_to_features(sequences)

        if not feature_matrix:
            print("No valid feature matrices created. Ensure the input files have valid opcodes.")
            return

        # Split the dataset into training and testing sets
        X_train, X_test = train_test_split(feature_matrix, test_size=0.2, random_state=42)

        # Train HMM model
        hmm_model = train_hmm_model(X_train, n_components=5)
        joblib.dump(hmm_model, model_file)
        print("Trained and saved HMM model to disk.")

        # Save the opcode_to_idx mapping for later use
        joblib.dump(opcode_to_idx, 'opcode_to_idx.pkl')
        print("Saved opcode_to_idx mapping to disk.")

        # Calculate log probabilities for training sequences to find an optimal threshold
        malware_log_probs = [classify_sequence(hmm_model, seq) for seq in X_train]
        print(f"Malware Log Probabilities: {malware_log_probs}")

        if malware_log_probs:
            threshold = 1e-16  # Lowering the threshold further to account for close cases
            print(f"Adjusted Threshold: {threshold}")
        else:
            print("No log probabilities calculated; skipping threshold adjustment.")
            threshold = 0.0  

    # Print recognized opcodes for debugging
    print(f"Recognized opcodes: {list(opcode_to_idx.keys())}")

    # Example of classifying new opcode sequences
    malicious_sequences = [
        ['add', 'mov', 'xor', 'jmp', 'call'],   # Example sequence
        ['mov', 'sub', 'add', 'jmp', 'dec'],
        ['call', 'add', 'sub', 'mov', 'jmp'],
        ['mov', 'jmp', 'dec', 'xor', 'ret']# Example sequence
    ]

    # Classifying malicious sequences
    for seq in malicious_sequences:
        classify_new_sequence(hmm_model, opcode_to_idx, seq, threshold)
        
    

if __name__ == "__main__":
    main()


MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


[['call', 'test', 'sub', 'add', 'or'], ['add', 'mov', 'cmp', 'jz', 'mov'], ['mov', 'add', 'mov', 'sub', 'xor'], ['mov', 'sub', 'add', 'cmp', 'jnz'], ['jmp', 'mov', 'add', 'and', 'mov'], ['mov', 'add', 'sub', 'call', 'add'], ['mov', 'mov', 'inc', 'add', 'dec'], ['jnz', 'jmp', 'call', 'mov', 'add'], ['add', 'cmp', 'jz', 'mov', 'add'], ['cmp', 'jz', 'cmp', 'jnz', 'jmp'], ['xor', 'dec', 'add', 'cmp', 'add'], ['cmp', 'jz', 'mov', 'add', 'cmp'], ['jz', 'cmp', 'jnz', 'jmp', 'mov'], ['and', 'cmp', 'jz', 'call', 'call'], ['mov', 'mov', 'add', 'call', 'add'], ['cmp', 'jz', 'call', 'mov', 'cmp'], ['jz', 'call', 'cmp', 'jnz', 'mov'], ['call', 'jmp', 'mov', 'mov', 'mov'], ['mov', 'cmp', 'jz', 'jmp', 'call'], ['mov', 'mov', 'add', 'call', 'add'], ['jz', 'xor', 'sub', 'call', 'cmp'], ['jz', 'sub', 'mov', 'sub', 'call'], ['cmp', 'jz', 'mov', 'mov', 'call'], ['call', 'call', 'call', 'mov', 'sub'], ['mov', 'add', 'mov', 'cmp', 'jnz'], ['jmp', 'cmp', 'jz', 'inc', 'sub'], ['inc', 'sub', 'jmp', 'add', 'mov